# Frontier Search

See [https://mathrepo.mis.mpg.de/PolynomialNeuralNetworks/](https://mathrepo.mis.mpg.de/PolynomialNeuralNetworks/) which provides the following code chunk. You can read there for details on how the algorithm works and other examples.

Running the search with for $h=8$, $max\_width=8$, $min\_width=1$ took roughly 170 min.

Running the search with $h=7$, $max\_width=7$, $min\_width=1$ took roughly 15 min.

In [1]:
# %load dim_backprop.py
from sage.all import *


class Network(object):

    def __init__(self, sizes, exponent=2, finite_field=100003):
        """The list ``sizes`` contains the number of neurons in the
        respective layers of the network.  For example, if the list
        was [2, 3, 1] then it would be a three-layer network, with the
        first layer containing 2 neurons, the second layer 3 neurons,
        and the third layer 1 neuron.  Currently, biases are initialized to 
        zero and weights are initialized randomly."""
        self.num_layers = len(sizes)
        self.sizes = sizes
        self.FF = GF(finite_field)
        self.biases = [zero_matrix(self.FF, y, 1) for y in sizes[1:]]  # [random_matrix(FF, y, 1) for y in sizes[1:]]
        self.weights = [random_matrix(self.FF, y, x) for x, y in zip(sizes[:-1], sizes[1:])]
        self.exponent = exponent
        self.degree = self.exponent**(self.num_layers - 2)

    def feedforward(self, a):
        """Return the output of the network if ``a`` is input."""
        for b, w in zip(self.biases[:-1], self.weights[:-1]):
            a = matrix_power(w * a + b, self.exponent)
        return self.weights[-1] * a + self.biases[-1]

    def backprop(self, x, pullback_vector=None):
        """Return a tuple ``(nabla_b, nabla_w)`` representing the
        gradient for the *output*pullback_vector* function.  ``nabla_b`` and
        ``nabla_w`` are layer-by-layer lists of matrices, similar
        to ``self.biases`` and ``self.weights``."""

        if pullback_vector is None:
            pullback_vector = ones_matrix(self.FF, self.sizes[-1], 1)

        nabla_b = [zero_matrix(self.FF, b.nrows(), b.ncols()) for b in self.biases]
        nabla_w = [zero_matrix(self.FF, w.nrows(), w.ncols()) for w in self.weights]
        # feedforward
        activation = x
        activations = [x]  # list to store all the activations, layer by layer
        zs = []  # list to store all the z vectors, layer by layer
        for b, w in zip(self.biases, self.weights):
            z = w * activation + b
            zs.append(z)
            activation = matrix_power(z, self.exponent)
            activations.append(activation)
        # backward pass
        delta = pullback_vector
        nabla_b[-1] = delta
        nabla_w[-1] = delta * activations[-2].T
        for l in range(2, self.num_layers):
            z = zs[-l]
            sp = matrix_power_prime(z, self.exponent)
            delta = elementwise_product(self.weights[-l + 1].T * delta, sp)
            nabla_b[-l] = delta
            nabla_w[-l] = delta * activations[-l - 1].T
        return (nabla_b, nabla_w)


def matrix_power(M, exponent=2):
    """Raise elements in M to the power exponent."""
    nc, nr = M.ncols(), M.nrows()
    A = copy(M.parent().zero())
    for r in range(nr):
        for c in range(nc):
            A[r, c] = M[r, c]**exponent
    return A


def matrix_power_prime(M, exponent=2):
    """Derivative of matrix_power."""
    nc, nr = M.ncols(), M.nrows()
    A = copy(M.parent().zero())
    for r in range(nr):
        for c in range(nc):
            A[r, c] = exponent * M[r, c]**(exponent - 1)
    return A


def elementwise_product(M, N):
    """Element-wise product of M and N."""
    nc, nr = M.ncols(), M.nrows()
    A = copy(M.parent().zero())
    for r in range(nr):
        for c in range(nc):
            A[r, c] = M[r, c] * N[r, c]
    return A


def monomial_list(v, k):
    """Return a list of all monomials in the entries of v of degree k."""
    nvars = len(v)
    exponents_list = list(WeightedIntegerVectors(k, [1 for t in v]))
    return [prod([v[i] ** exponents[i] for i in range(nvars)]) for exponents in exponents_list]


def compute_dimension(network_widths, network_exponent):
    """Compute the dimension of the neurovariety of a network with arbitrary output dimension."""

    primes = [100003, 100153]   # run the computation over GF(p) for p in primes
    dims = []
    for p in primes:
        nn = Network(network_widths, network_exponent, p)
        num_params = sum([m * n for m, n in zip(nn.sizes[:-1], nn.sizes[1:])])
        degree = nn.degree
        dim_poly_vector = binomial(nn.degree + nn.sizes[0] - 1, nn.sizes[0] - 1)
        nsamples = 2 * dim_poly_vector  # nsamples should be at least dim_poly_vector
        x = random_matrix(nn.FF, nn.sizes[0], nsamples)
        monomials = matrix(nn.FF, [monomial_list(v, degree) for v in x.T])
        monomials_pinv = monomials.pseudoinverse()
        jacobian_matrix = zero_matrix(nn.FF, nn.sizes[-1] * dim_poly_vector, num_params)
        for j in range(nn.sizes[-1]):
            gradients_samples = zero_matrix(nn.FF, nsamples, num_params)
            basis_vec = zero_matrix(nn.FF, nn.sizes[-1], 1)
            basis_vec[j, 0] = 1
            for i in range(nsamples):
                gradient_matrices = nn.backprop(x[:, i], basis_vec)[1]  # use no biases
                gradients_samples[i, :] = matrix(nn.FF, [[t for mat in gradient_matrices for t in mat.list()]])  
            jacobian_matrix[j * dim_poly_vector:(j + 1) * dim_poly_vector, :] = monomials_pinv * gradients_samples
        dims.append(rank(jacobian_matrix))
    if not all(d == dims[0] for d in dims):
        raise ValueError('different dimensions over finite fields: ' + str(dims))
    ambient_dim = (binomial(nn.degree + nn.sizes[0] - 1, nn.sizes[0] - 1)) * nn.sizes[-1]
    naive_bound = sum([(m - 1) * n for m, n in zip(network_widths[:-1], network_widths[1:])]) + network_widths[-1]
    ex_dim = min(ambient_dim, naive_bound)
    return( nn.sizes, # d
            nn.exponent,                         # r
            ambient_dim,                         # ambient_dim
            ex_dim,                              # expected_dim
            dims[0],                           # dimension 
            ex_dim - dims[0]                   # defect
           )

# Search Algorithm


Here's a very brief explanation of the search algorithm.

1. Fix $d_0,d_h,r$. User specifies the max_width and min_width for the hidden layers.

2. Generate all the possible architectures subject to the max_width and min_width constraints. We also delete from the list of possibilties all architectures whose expected dimension is less than the ambient dimension.

3. User can add guesses to check first. If not, the algorithm generates randomly choices among the valid possible architectures which we denote the set of as $\mathcal{S}$.

4. Test a random element of $\mathcal{S}$ or the user specified guess. Two things can happen.

    a. It is filling. Then delete from $\mathcal{S}$ all architectures $\geq$ than the one just tested.

    b. It is not filling (aka short). Then delete from $\mathcal{S}$ all architectures $<$ the one just tested.

5. Repeat step 4. until $\mathcal{S}$ consists only of architectures we've already tested. These are the minimal filling architectures.

In [2]:
import os
import ast
import math
import itertools
import random as py_random
import pandas as pd
from tqdm import tqdm

In [3]:
# Helper Functions

def calculate_parameter_count(hidden: tuple, d_0: int, d_h: int) -> int:
    """Calculates the total number of weights and biases in the network."""
    sizes = [d_0] + list(hidden) + [d_h]
    return sum(m * n for m, n in zip(sizes[:-1], sizes[1:]))

def is_less_or_equal(t1: tuple, t2: tuple) -> bool:
    """Returns True if every element in t1 is <= the corresponding element in t2."""
    if len(t1) != len(t2):
        return False
    return all(a <= b for a, b in zip(t1, t2))

def evaluate_single_architecture(hidden_tuple: tuple, h: int, d_0: int, d_h: int, exponent: int):
    """Worker function to evaluate a single architecture and format the result."""
    sizes = [d_0] + list(hidden_tuple) + [d_h]
    arch_str = str(sizes)
    
    try:
        _, _, amb, _, dim, _ = compute_dimension(sizes, exponent)
        params = calculate_parameter_count(hidden_tuple, d_0, d_h)
        is_full = (dim == amb)
        
        status = "FULL " if is_full else "SHORT"
        print(f"  [{status}] {arch_str} -> Rank: {dim}/{amb} (Params: {params})")
        
        return {
            "h": h,
            "exponent": exponent,
            "architecture": arch_str,
            "num_parameters": params,
            "dimension_computed": int(dim),
            "ambient_dimension": int(amb),
            "is_full_dimension": is_full,
            "is_minimal": False 
        }
    except Exception as e:
        print(f"  [ERROR] {arch_str} failed: {e}")
        return None


In [4]:
# Core Search & Pruning Algorithm

def parameter_boundary_search(
    h_values: list, max_width=6, min_width=1, exponent=2, 
    d_0=2, d_h=1, csv_filename="architecture_search_log.csv", user_guesses=None
):
    # Set-up a warm-up computation...if this breaks then there's no point of running the rest.
    print("Warming up dimension computation...")
    compute_dimension([d_0, 2, d_h], exponent)
    print("Warm-up complete...\n")

    # Load or initialize database -- storing in .csv for readability too
    if os.path.exists(csv_filename):
        print(f"Loading existing database from '{csv_filename}'...")
        df = pd.read_csv(csv_filename)
    else:
        print("No existing database found. Starting fresh...")
        df = pd.DataFrame(columns=[
            "h", "exponent", "architecture", "num_parameters", 
            "dimension_computed", "ambient_dimension", "is_full_dimension", "is_minimal"
        ])

    evaluated_architectures = set(df["architecture"].tolist())
    new_records = []

    for h in h_values:
        print(f"\n--- FRONTIER SEARCH: h={h}, exponent={exponent} (Max Width: {max_width} & Min Width: {min_width})  ---")
        
        num_hidden = h - 1
        degree = exponent ** num_hidden
        ambient_dim = math.comb(degree + d_0 - 1, d_0 - 1) * d_h
        print(f"  Target Ambient Dimension: {ambient_dim}")

        # Extract known boundaries
        known_minimal = set()
        known_short = set()
        
        if not df.empty:
            subset_df = df[(df["h"] == h) & (df["exponent"] == exponent)]
            
            min_df = subset_df[subset_df["is_minimal"] == True]
            known_minimal.update(tuple(ast.literal_eval(a)[1:-1]) for a in min_df["architecture"])
            
            short_df = subset_df[subset_df["is_full_dimension"] == False]
            known_short.update(tuple(ast.literal_eval(a)[1:-1]) for a in short_df["architecture"])


        # 1. Evaluate User Guesses

        if user_guesses and h in user_guesses:
            for guess in user_guesses[h]:
                arch_str = str([d_0] + list(guess) + [d_h])
                if arch_str in evaluated_architectures:
                    continue
                
                print(f"  Evaluating Guess: {guess}...")
                res = evaluate_single_architecture(guess, h, d_0, d_h, exponent)
                if not res: continue
                
                evaluated_architectures.add(res["architecture"])
                new_records.append(res)
                
                if res["is_full_dimension"]:
                    known_minimal.add(guess)
                else:
                    known_short.add(guess)

        # 2. Build and Filter Candidate Pool
        # This will filter using known architectures and the trivial bound

        total_combinations = (max_width + 1 - max(1, min_width)) ** num_hidden
        print("  Generating candidate pool...")
        all_possible = itertools.product(range(max(1, min_width), max_width + 1), repeat=num_hidden)
        print("  Obtained initial pool...")
        candidate_pool = []
        rejected_count = 0
    
        for hidden in tqdm(all_possible, total=total_combinations, desc="Evaluating Architectures"):
            arch_str = str([d_0] + list(hidden) + [d_h])
            if arch_str in evaluated_architectures:
                continue
                
            params = calculate_parameter_count(hidden, d_0, d_h)
            
            # pruning obvious noncandidates
            if (params < ambient_dim or 
                any(is_less_or_equal(m, hidden) for m in known_minimal) or 
                any(is_less_or_equal(hidden, s) for s in known_short)):
                rejected_count += 1
                continue
                
            candidate_pool.append(hidden)
    
        print(f"  Pruned {rejected_count} impossible/redundant architectures.")
        print(f"  Starting sequential evaluation on {len(candidate_pool)} viable candidates.")
        

        # Shuffle queue
        py_random.shuffle(candidate_pool)
        print(f"  Finished shuffling...")


    # 3. evaluation loop
    # not currently written so I can interrupt midway through and be okay
        while candidate_pool:

            target = candidate_pool.pop()

            if any(is_less_or_equal(m, target) for m in known_minimal):
                continue
            if any(is_less_or_equal(target, s) for s in known_short):
                continue
            
            res = evaluate_single_architecture(target, h, d_0, d_h, exponent)

            if not res: 
                continue
            
            evaluated_architectures.add(res["architecture"])
            new_records.append(res)
            
            # Simply update the known sets. The lazy pruning checks above 
            # will naturally catch redundant supersets/subsets on future iterations.

            if res["is_full_dimension"]:
                known_minimal.add(target)
            else:
                known_short.add(target)       

    # 4. Save and Post-Process Minimality
    if new_records:
        new_df = pd.DataFrame(new_records).dropna(axis=1, how='all')
        df = pd.concat([df, new_df], ignore_index=True)
        print(f"\nAdded {len(new_records)} new architectures to the database.")

    print("Re-evaluating minimal filling properties (grouped by depth h and exponent)...")


    df["is_minimal"] = False 

    # Using the the know architectures, check for minimality among all possiblities

    for (h_val, exp_val), group in df[df["is_full_dimension"] == True].groupby(["h", "exponent"]):
        
        full_archs = [ast.literal_eval(arch) for arch in group["architecture"]]
        
        for idx, row in group.iterrows():

            parsed_config = ast.literal_eval(row["architecture"])

            # It is minimal if NO other full architecture is strictly smaller than it
            
            is_min = not any(
                other != parsed_config and is_less_or_equal(other, parsed_config) 
                for other in full_archs
            )
            
            df.at[idx, "is_minimal"] = is_min

    df.to_csv(csv_filename, index=False)
    print(f"Database successfully saved to '{csv_filename}'.\n")
    return df

In [5]:
if __name__ == "__main__":

    h_values_to_test = [2,3,4,5,6,7]  
    
    # Optional: Provide known architectures to jumpstart the pruning process.
    my_guesses = {}
        
    d0=2
    dh=1

    # Run the frontier search
    df_results = parameter_boundary_search(
        h_values=h_values_to_test, 
        max_width=7,        # Maximum width 
        min_width=2,        # Minimum width
        exponent=2,         # The activation degree
        d_0=d0,              # Input dimension
        d_h=dh,              # Output dimension
        csv_filename=f"{d0}_{dh}_architectures.csv",
        user_guesses=my_guesses
    )
    
    # Display the minimal architectures found so far
    print("\n=== CURRENT MINIMAL FILLING ARCHITECTURES IN DATABASE ===")
    minimal_archs = df_results[df_results['is_minimal'] == True]
    
    if not minimal_archs.empty:
        # Sort by parameters for easier reading
        minimal_archs = minimal_archs.sort_values(by="num_parameters")
        print(minimal_archs[["architecture", "num_parameters", "dimension_computed"]].to_string(index=False))
    else:
        print("No minimal full architectures found matching the criteria.")

Warming up dimension computation...
Warm-up complete...

Loading existing database from '2_1_architectures.csv'...

--- FRONTIER SEARCH: h=2, exponent=2 (Max Width: 7 & Min Width: 2)  ---
  Target Ambient Dimension: 3
  Generating candidate pool...
  Obtained initial pool...


Evaluating Architectures: 100%|██████████| 6/6 [00:00<00:00, 10681.59it/s]

  Pruned 5 impossible/redundant architectures.
  Starting sequential evaluation on 0 viable candidates.
  Finished shuffling...

--- FRONTIER SEARCH: h=3, exponent=2 (Max Width: 7 & Min Width: 2)  ---
  Target Ambient Dimension: 5


  Generating candidate pool...
  Obtained initial pool...


Evaluating Architectures: 100%|██████████| 36/36 [00:00<00:00, 51411.28it/s]


...

  Pruned 1260 impossible/redundant architectures.
  Starting sequential evaluation on 0 viable candidates.
  Finished shuffling...

--- FRONTIER SEARCH: h=6, exponent=2 (Max Width: 7 & Min Width: 2)  ---
  Target Ambient Dimension: 33
  Generating candidate pool...
  Obtained initial pool...


Evaluating Architectures: 100%|██████████| 7776/7776 [00:00<00:00, 35155.94it/s]


  Pruned 7696 impossible/redundant architectures.
  Starting sequential evaluation on 0 viable candidates.
  Finished shuffling...

--- FRONTIER SEARCH: h=7, exponent=2 (Max Width: 7 & Min Width: 2)  ---
  Target Ambient Dimension: 65
  Generating candidate pool...
  Obtained initial pool...


Evaluating Architectures: 100%|██████████| 46656/46656 [00:03<00:00, 13012.57it/s]


  Pruned 46455 impossible/redundant architectures.
  Starting sequential evaluation on 0 viable candidates.
  Finished shuffling...
Re-evaluating minimal filling properties (grouped by depth h and exponent)...
Database successfully saved to '2_1_architectures.csv'.


=== CURRENT MINIMAL FILLING ARCHITECTURES IN DATABASE ===
                architecture  num_parameters  dimension_computed
                   [2, 2, 1]               6                   3
                [2, 2, 2, 1]              10                   5
             [2, 3, 3, 2, 1]              23                   9
          [2, 3, 3, 3, 2, 1]              32                  17
       [2, 3, 3, 4, 4, 2, 1]              53                  33
    [2, 3, 3, 4, 6, 5, 2, 1]              93                  65
    [2, 3, 3, 4, 5, 6, 3, 1]              98                  65
    [2, 3, 4, 5, 5, 5, 2, 1]             100                  65
    [2, 3, 4, 5, 6, 4, 2, 1]             102                  65
    [2, 3, 3, 5, 7, 4, 2